# Ex 1

1. xor intre 2 variabile identice da mereu 0 -> nu e PRNG
2. creste liniar, distributie neuniforma -> nu e PRNG
3. ajunge foarte repede la 0 si se blocheaza, distributie neuniforma -> nu e PRNG

# Ex 2

In [10]:
import secrets
import string

# a - generarea automata a unei parole sigure
def generate_password(length=10):
    if length < 10:
        raise ValueError("Length needs to be at least 10")

    specials = ".!$@"
    
    upper = secrets.choice(string.ascii_uppercase)
    digit = secrets.choice(string.digits)
    special = secrets.choice(specials)
    
    pool = string.ascii_letters + string.digits + specials
    remaining = [secrets.choice(pool) for _ in range(length - 3)]
    
    pwd_list = [upper, digit, special] + remaining
    secrets.SystemRandom().shuffle(pwd_list)
    
    return ''.join(pwd_list)

print(generate_password())

!zTRg9xSzu


In [14]:
#b - generare token-uri de resetare de parola / recuperare cont
def generate_url_safe_token(length=32):
    if length < 32:
        raise ValueError("Min len needs to be 32")
    token = secrets.token_urlsafe(length)
    return token[:length]

generate_url_safe_token()


'Q4eiw_uBRcKJPTip94O09IiR9yIPxFJm'

In [22]:
# c - generare tokene hex pentru identificarea unei sesiuni
def generate_hex_token(length=32):
    if length < 32:
        raise ValueError("Min len needs to be 32")
    
    byte_count = length // 2
    token = secrets.token_hex(byte_count)
    
    return token

generate_hex_token()

'ed1ec1d728b09fdc08b3d71bd5cac3f8'

In [25]:
# d 
import unicodedata
from typing import Union

def const_time_equal(a: Union[str, bytes], b: Union[str, bytes], normalize_unicode: bool = True) -> bool:
    if isinstance(a, str) and isinstance(b, str):
        if normalize_unicode:
            a = unicodedata.normalize("NFC", a)
            b = unicodedata.normalize("NFC", b)
        a = a.encode("utf-8")
        b = b.encode("utf-8")
    elif isinstance(a, bytes) and isinstance(b, bytes):
        pass
    else:
        raise TypeError("Arguments are not of the same type")

    return secrets.compare_digest(a, b)

token1 = secrets.token_bytes(32)
token2 = token1[:]
print(const_time_equal(token1, token2))

s1 = "café"
s2 = "cafe\u0301"
print(const_time_equal(s1, s2))

True
True


In [26]:
# e
def generate_fluid_binary_key(length=100):
    return secrets.token_bytes(length)

print(generate_fluid_binary_key())

b"\x80\x1c\xd7\x81\xeb_\x16\x91\xc1\x97\xe2o\xd7W[\xc9uA\x02$\x8f\xd3\xb4\xde^\xe7S\xbf\x99\x9e\x1e+fp\x98\xcc\xbf\r\x11\xf7\xf4\x07\x16\xe2\xde\xc7\x0b\xfd\xb9\xa8\x1a\x0e\xe5X\x9bF\xca\xe8^a9\xfe&0\x89i\x05\x11c\xfd\x84\x11\xe7\xfa\xee%'\xf6Y9,\x8a@2;\xb1\xc1\xf7\xcd8M\xfaG$\x81S\x9ex9\xa1"


In [28]:
# f - Am folosit bcrypt, deoarece are rezistenta la atacuri brute force, are password salting automat si este rapid pentru ca e scris in Rust.
import bcrypt

def store_password(plain_password: str) -> bytes:
    return bcrypt.hashpw(plain_password.encode(), bcrypt.gensalt())

def verify_password(plain_password: str, stored_hash: bytes) -> bool:
    return bcrypt.checkpw(plain_password.encode(), stored_hash)

stored = store_password("parola_secur1zata!!")
print(stored) 

print(verify_password("parola_secur1zata!!!", stored))
print(verify_password("parola_naspa", stored))

b'$2b$12$0hCJ7gprJNnapNF0l5I0Cu61TEkQEV6yiYdO2InBe6ntP0r30bgX2'
False
False


# Ex 3

- Cod Java:
Seed constant -> secventa de numere generate va fi aceeasi la fiecare rulare

Cod PHP:
Se poate reconstitui sessionID, daca se stie userID 

- Java: CWE-336 - Same Seed in PRNG
PHP: CWE-335 - Incorrect Usage of Seeds in PRNG

- Daca spatiul seed-urilor este redus -> vulnerabilitate la brute force
Cod CWE asociat: CWE-339 - Small Seed Space in PRNG

- CAPEC-112 - Brute Force

- CWE-338 - Use of Cryptographically Weak PRNG
CWE-332 - Insufficient Entropy in PRNG
CWE-1241 - Use of Predictable Algorithm in PRNG

**CVE-2025-62710**:

**Title**: Sakai kernel-impl: predictable PRNG used to generate server‑side encryption key in EncryptionUtilityServiceImpl

**Description**:
"Sakai is a Collaboration and Learning Environment. Prior to versions 23.5 and 25.0, EncryptionUtilityServiceImpl initialized an AES256TextEncryptor password (serverSecretKey) using RandomStringUtils with the default java.util.Random. java.util.Random is a non‑cryptographic PRNG and can be predicted from limited state/seed information (e.g., start time window), substantially reducing the effective search space of the generated key. An attacker who can obtain ciphertexts (e.g., exported or at‑rest strings protected by this service) and approximate the PRNG seed can feasibly reconstruct the serverSecretKey and decrypt affected data. SAK-49866 is patched in Sakai 23.5, 25.0, and trunk."

6. 21 de vulnerabilitati in 2025